# Submit SCF energy calculation

In [ ]:
%%javascript
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}

In [ ]:
# General imports.
import urllib.parse as urlparse

import ipywidgets as ipw
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output

# AiiDA imports.
%load_ext aiida
%aiida
# AiiDAlab imports.
import aiidalab_widgets_base as awb
import aiidalab_widgets_empa as awe
from aiida import orm, plugins

from surfaces_tools.utils import wfn

# Custom imports.
from surfaces_tools.widgets import editors, inputs



In [ ]:
Cp2kScfWorkChain = plugins.WorkflowFactory("nanotech_empa.cp2k.scf")

In [ ]:
# Structure selector.
# SCF keeps atom positions fixed, so geometry-optimization constraints are hidden here.
inputs.SECTIONS_TO_DISPLAY = {
    name: [section for section in sections if section is not inputs.constraints.ConstraintsWidget]
    for name, sections in inputs.SECTIONS_TO_DISPLAY.items()
}
build_slab = editors.BuildSlab(title="Build slab")
input_details = inputs.InputDetails()

structure_selector = awb.StructureManagerWidget(
    importers=[
        awb.StructureUploadWidget(title="Import from computer"),
        awb.StructureBrowserWidget(title="AiiDA database"),
        awb.SmilesWidget(title="From SMILES"),
        awe.CdxmlUploadWidget(title="CDXML"),
    ],
    editors=[
        awb.BasicStructureEditor(title="Edit structure"),
        build_slab,
        awb.BasicCellEditor(),
        editors.InsertStructureWidget(title="Insert molecule"),
    ],
    storable=True,
    node_class="StructureData",
)
ipw.dlink((structure_selector, "structure"), (build_slab, "molecule"))
ipw.dlink((structure_selector, "structure"), (input_details, "structure"))
ipw.dlink((structure_selector, "structure_node"), (input_details, "structure_node"))
ipw.dlink((input_details, "details"), (build_slab, "details"))
display(structure_selector)

# Code.
code_input_widget = awb.ComputationalResourcesWidget(
    description="CP2K code:", default_calc_job_plugin="cp2k"
)
sparse_overlap_code = awb.ComputationalResourcesWidget(
    description="Overlap converter:",
    default_calc_job_plugin="nanotech_empa.sparse_overlap",
)
unfolding_code = awb.ComputationalResourcesWidget(
    description="Unfolding code:",
    default_calc_job_plugin="nanotech_empa.cp2k_unfolding",
)
bader_code = awb.ComputationalResourcesWidget(
    description="Bader code:",
    default_calc_job_plugin="nanotech_empa.bader",
)
resources = awe.ProcessResourcesWidget()
unfolding_resources = awe.ProcessResourcesWidget()

output = ipw.Output()



In [ ]:
url = urlparse.urlsplit(jupyter_notebook_url)
parsed_url = urlparse.parse_qs(url.query)
if "structure_uuid" in parsed_url:
    structure_selector.input_structure = load_node(parsed_url["structure_uuid"][0])

In [ ]:
# Protocol.
protocol = ipw.Dropdown(
    value="standard",
    options=[
        ("Standard", "standard"),
        ("Low accuracy", "low_accuracy"),
        ("Debug", "debug"),
    ],
    description="Protocol:",
    style={"description_width": "initial"},
)

write_overlap_matrix = ipw.Checkbox(
    value=False,
    description="Write overlap matrix",
    style={"description_width": "initial"},
)

compute_unfolding = ipw.Checkbox(
    value=False,
    description="Compute band unfolding",
    style={"description_width": "initial"},
)

compute_bader_charges = ipw.Checkbox(
    value=False,
    description="Compute Bader charges",
    style={"description_width": "initial"},
)

bader_cutoff = ipw.FloatText(
    value=1200.0,
    description="Bader cutoff:",
    style={"description_width": "initial"},
    layout={"width": "240px"},
)

diagonalisation_smearing = inputs.DiagonalisationSmearingWidget()

added_mos = ipw.BoundedIntText(
    value=100,
    min=0,
    max=100000,
    description="ADDED_MOS:",
    style={"description_width": "initial"},
    layout={"width": "220px"},
)

overlap_ndigits = ipw.BoundedIntText(
    value=14,
    min=1,
    max=30,
    description="Overlap digits:",
    style={"description_width": "initial"},
    layout={"width": "220px"},
)

retrieve_sparse_overlap = ipw.Checkbox(
    value=False,
    description="Also retrieve sparse overlap matrix",
    style={"description_width": "initial"},
)

overlap_threshold = ipw.FloatText(
    value=1.0e-10,
    description="AO threshold:",
    style={"description_width": "initial"},
    layout={"width": "240px"},
)

unfolding_a1 = ipw.Text(description="a1:", style={"description_width": "initial"}, layout={"width": "70%"})
unfolding_a2 = ipw.Text(description="a2:", style={"description_width": "initial"}, layout={"width": "70%"})
unfolding_a3 = ipw.Text(description="a3:", placeholder="Leave empty for 2D; leave a2/a3 empty for 1D", style={"description_width": "initial"}, layout={"width": "70%"})
unfolding_lattice_type = ipw.Dropdown(
    value="auto",
    options=["auto", "1d", "square", "rectangular", "hexagonal", "oblique"],
    description="Path type:",
    style={"description_width": "initial"},
)
unfolding_path = ipw.Text(value="G-K-M-G", description="Path:", style={"description_width": "initial"}, layout={"width": "260px"})
unfolding_primitive_basis_atoms = ipw.Text(
    value="",
    description="Basis atoms:",
    placeholder="1-based indices, e.g. 1 2 or 171 196",
    style={"description_width": "initial"},
    layout={"width": "420px"},
)
unfolding_basis_cluster_tol = ipw.FloatText(
    value=5.0e-2,
    description="Basis tol:",
    style={"description_width": "initial"},
    layout={"width": "220px"},
)
unfolding_preview = ipw.Textarea(value="", description="snapped:", disabled=True, rows=7, style={"description_width": "initial"}, layout={"width": "70%"})
unfolding_kmap_output = ipw.Output()
unfolding_help_toggle = ipw.Checkbox(value=False, description="Show unfolding help", style={"description_width": "initial"})
unfolding_help = ipw.HTML(
    value=(
        "<div style='max-width: 760px'>"
        "<p><b>Primitive vectors.</b> Enter approximate primitive-cell vectors in Angstrom. "
        "Use a1 for 1D, a1/a2 for 2D, and a1/a2/a3 for 3D-style input. "
        "Vectors can come from clicked atom-to-atom displacements; small relaxation errors are expected.</p>"
        "<p><b>Exact tiling.</b> The interface snaps those approximate vectors to the closest exact "
        "integer tiling of the selected supercell, S = A M. The immutable snapped vectors and integer "
        "matrix M shown below are the ones submitted.</p>"
        "<p><b>Primitive basis atoms.</b> Enter 1-based atom indices that define one representative "
        "primitive basis, for example one B and one N atom for hBN. The code uses the snapped "
        "primitive vectors, translates these basis atoms through the supercell, assigns every atom "
        "to the nearest matching basis site, and stores max/mean displacement diagnostics.</p>"
        "<p><b>Basis tolerance.</b> This is only a fallback for automatic clustering when basis atoms "
        "are omitted. For production unfolding, provide explicit primitive basis atoms.</p>"
        "<p><b>Gamma unfolding.</b> A Gamma-only supercell calculation corresponds to primitive-cell "
        "k-points satisfying M<sup>T</sup> q = integer. The preview shows all folded k-points and "
        "highlights which ones lie exactly on the selected path.</p>"
        "<p><b>Unfolding weight.</b> Each supercell Kohn-Sham orbital "
        "&psi;<sub>i</sub> at energy &epsilon;<sub>i</sub> is projected onto the "
        "primitive-cell Bloch AO subspace at each folded point <b>q</b>. "
        "The vector <b>c</b><sub>i</sub> contains the AO coefficients of &psi;<sub>i</sub>; "
        "<b>B</b><sub>q</sub> is the matrix that forms the primitive-cell Bloch AO combinations at <b>q</b> "
        "from the AOs in all repeated primitive cells of the supercell. "
        "In the non-orthogonal AO basis we use the AO overlap matrix <b>S</b>: "
        "construct the Bloch AO metric "
        "<b>S</b><sub>q</sub> = <b>B</b><sub>q</sub><sup>&dagger;</sup> "
        "<b>S</b> <b>B</b><sub>q</sub> and the overlap vector "
        "<b>v</b><sub>qi</sub> = <b>B</b><sub>q</sub><sup>&dagger;</sup> "
        "<b>S</b> <b>c</b><sub>i</sub>. The plotted spectral weight is "
        "w<sub>i</sub>(<b>q</b>) = <b>v</b><sub>qi</sub><sup>&dagger;</sup> "
        "<b>S</b><sub>q</sub><sup>-1</sup> <b>v</b><sub>qi</sub>. "
        "Thus the same eigenvalue &epsilon;<sub>i</sub> appears at every allowed "
        "primitive <b>q</b>, with marker size proportional to w<sub>i</sub>(<b>q</b>).</p>"
        "<p><b>Post-processing.</b> After the diagonalization CP2K step, AiiDA runs a remote post-processing "
        "calculation using the WFN file and AO overlap matrix in the CP2K remote folder. It retrieves only "
        "compact unfolding weights and eigenvalues for fast plotting.</p>"
        "</div>"
    )
)
unfolding_help_box = ipw.VBox([])

sparse_retrieve_options = ipw.VBox([])
overlap_options = ipw.VBox([])
unfolding_options = ipw.VBox([])
bader_options = ipw.VBox([])


def _vector_to_text(vector):
    return " ".join(f"{x:.10g}" for x in vector)


def _parse_vector(text):
    text = text.strip()
    if not text:
        return None
    values = [float(x) for x in text.replace(",", " ").split()]
    if len(values) != 3:
        raise ValueError("Each vector must contain three numbers.")
    return np.asarray(values, dtype=float)


def _read_unfolding_vectors():
    vectors = [_parse_vector(w.value) for w in (unfolding_a1, unfolding_a2, unfolding_a3)]
    if vectors[0] is None:
        raise ValueError("a1 is required.")
    if vectors[1] is None and vectors[2] is not None:
        raise ValueError("a2 cannot be empty when a3 is set.")
    return np.asarray([v for v in vectors if v is not None], dtype=float)



def _parse_atom_indices(text, natoms=None):
    items = []
    clean = str(text).replace(",", " ").strip()
    if not clean:
        return []
    for token in clean.split():
        if ".." in token:
            left, right = token.split("..", 1)
            start = int(left)
            stop = int(right)
            step = 1 if stop >= start else -1
            items.extend(range(start, stop + step, step))
        elif "-" in token and token.count("-") == 1 and not token.startswith("-"):
            left, right = token.split("-", 1)
            start = int(left)
            stop = int(right)
            step = 1 if stop >= start else -1
            items.extend(range(start, stop + step, step))
        else:
            items.append(int(token))
    if any(index < 1 for index in items):
        raise ValueError("Basis atom indices are 1-based and must be positive.")
    if natoms is not None and any(index > natoms for index in items):
        raise ValueError(f"Basis atom index exceeds number of atoms ({natoms}).")
    if len(set(items)) != len(items):
        raise ValueError("Basis atom indices contain duplicates.")
    return [index - 1 for index in items]


def _normalize_kind_symbol(symbol):
    text = str(symbol).strip()
    if not text:
        return text
    if len(text) >= 2 and text[1].islower():
        return text[:2]
    return text[:1]


def _mapping_displacement_summary(structure, primitive_vectors, basis_atom_text):
    basis_indices = _parse_atom_indices(basis_atom_text, natoms=len(structure.sites))
    if not basis_indices:
        return None
    dim = primitive_vectors.shape[0]
    atoms = structure.get_ase()
    positions = np.asarray(atoms.positions, dtype=float)
    symbols = [_normalize_kind_symbol(sym) for sym in atoms.get_chemical_symbols()]
    frac = np.linalg.solve(_lattice_matrix(primitive_vectors), positions[:, :dim].T).T
    basis_frac = frac[basis_indices] - np.floor(frac[basis_indices])
    basis_symbols = [symbols[index] for index in basis_indices]
    missing = sorted(set(symbols) - set(basis_symbols))
    if missing:
        raise ValueError("Primitive basis atoms do not cover elements: " + ", ".join(missing))
    A = _lattice_matrix(primitive_vectors)
    displacements = []
    atom_to_basis = []
    for sym, f in zip(symbols, frac):
        best = None
        for ibasis, (bsym, bf) in enumerate(zip(basis_symbols, basis_frac)):
            if sym != bsym:
                continue
            n = np.rint(f - bf).astype(int)
            residual = f - bf - n
            residual_cart = A @ residual
            dist = float(np.linalg.norm(residual_cart))
            if best is None or dist < best[0]:
                best = (dist, ibasis)
        if best is None:
            raise ValueError(f"No primitive basis atom with symbol {sym!r}")
        displacements.append(best[0])
        atom_to_basis.append(best[1])
    displacements = np.asarray(displacements, dtype=float)
    return {
        "basis_indices": np.asarray(basis_indices, dtype=int) + 1,
        "max": float(displacements.max()),
        "mean": float(displacements.mean()),
        "worst_atom": int(np.argmax(displacements)) + 1,
        "basis_counts": np.bincount(np.asarray(atom_to_basis, dtype=int), minlength=len(basis_indices)),
    }

def _lattice_matrix(vectors):
    dim = vectors.shape[0]
    return vectors[:dim, :dim].T


def _snap_vectors(approx_vectors, supercell_vectors):
    dim = approx_vectors.shape[0]
    mf = np.linalg.solve(_lattice_matrix(approx_vectors), _lattice_matrix(supercell_vectors))
    center = np.rint(mf).astype(int)
    best = None
    for delta in np.ndindex(*([3] * (dim * dim))):
        matrix = center + np.asarray(delta, dtype=int).reshape(dim, dim) - 1
        if abs(np.linalg.det(matrix)) < 0.5:
            continue
        primitive_lattice = _lattice_matrix(supercell_vectors) @ np.linalg.inv(matrix)
        snapped = np.zeros((dim, 3))
        snapped[:, :dim] = primitive_lattice.T
        score = float(np.linalg.norm(snapped - approx_vectors))
        item = (score, matrix, snapped, mf)
        if best is None or item[0] < best[0]:
            best = item
    if best is None:
        raise ValueError("Could not snap vectors to the supercell.")
    return best


def _reciprocal_vectors(primitive_vectors):
    return 2.0 * np.pi * np.linalg.inv(_lattice_matrix(primitive_vectors)).T


def _canonical_frac(values, tol=1.0e-8):
    folded = np.mod(np.asarray(values, dtype=float), 1.0)
    folded[np.isclose(folded, 1.0, atol=tol) | np.isclose(folded, 0.0, atol=tol)] = 0.0
    return folded


def _kfrac_to_cart(k_frac, primitive_vectors):
    k_frac = np.asarray(k_frac, dtype=float)
    B = _reciprocal_vectors(primitive_vectors)
    dim = B.shape[0]
    k_dim = k_frac @ B.T
    k_cart = np.zeros((len(k_dim), 3))
    k_cart[:, :dim] = k_dim
    return k_cart


def _clip_polygon_halfplane(poly, normal, offset, tol=1.0e-12):
    if len(poly) == 0:
        return poly
    clipped = []
    prev = poly[-1]
    prev_inside = np.dot(prev, normal) <= offset + tol
    for curr in poly:
        curr_inside = np.dot(curr, normal) <= offset + tol
        if curr_inside != prev_inside:
            denom = float(np.dot(curr - prev, normal))
            if abs(denom) > tol:
                t = float((offset - np.dot(prev, normal)) / denom)
                clipped.append(prev + t * (curr - prev))
        if curr_inside:
            clipped.append(curr)
        prev = curr
        prev_inside = curr_inside
    return np.asarray(clipped)


def _wigner_seitz_cell_2d(recip_vectors):
    b1, b2 = np.asarray(recip_vectors[:2, :2], dtype=float)
    neighbors = []
    for i in range(-2, 3):
        for j in range(-2, 3):
            if i == 0 and j == 0:
                continue
            g = i * b1 + j * b2
            neighbors.append(g)
    neighbors = sorted(neighbors, key=lambda g: np.dot(g, g))
    radius = 2.0 * max(np.linalg.norm(b1), np.linalg.norm(b2), 1.0)
    poly = np.asarray([[-radius, -radius], [radius, -radius], [radius, radius], [-radius, radius]], dtype=float)
    for g in neighbors:
        poly = _clip_polygon_halfplane(poly, g, 0.5 * float(np.dot(g, g)))
    center = poly.mean(axis=0)
    order = np.argsort(np.arctan2(poly[:, 1] - center[1], poly[:, 0] - center[0]))
    return poly[order]


def _draw_arrow(ax, vector, label, color):
    vector = np.asarray(vector, dtype=float)[:2]
    ax.arrow(0.0, 0.0, vector[0], vector[1], head_width=0.04 * max(np.linalg.norm(vector), 1.0),
             length_includes_head=True, color=color, linewidth=1.4)
    ax.text(vector[0], vector[1], label, color=color, ha="left", va="bottom")


def _pad_axes(ax, points):
    points = np.asarray(points, dtype=float)
    if points.size == 0:
        return
    xmin, ymin = points.min(axis=0)
    xmax, ymax = points.max(axis=0)
    pad = 0.08 * max(xmax - xmin, ymax - ymin, 1.0)
    ax.set_xlim(xmin - pad, xmax + pad)
    ax.set_ylim(ymin - pad, ymax + pad)


def _plot_lattice_diagnostics(ax_real, ax_rec, primitive_vectors, supercell_vectors, hs_points, path_labels, k_frac, path_q):
    avec = np.asarray(primitive_vectors, dtype=float)[:2, :2]
    svec = np.asarray(supercell_vectors, dtype=float)[:2, :2]
    bvec = _reciprocal_vectors(primitive_vectors)[:2, :2].T

    real_cell = np.asarray([[0, 0], avec[0], avec[0] + avec[1], avec[1], [0, 0]], dtype=float)
    super_cell = np.asarray([[0, 0], svec[0], svec[0] + svec[1], svec[1], [0, 0]], dtype=float)
    ax_real.plot(real_cell[:, 0], real_cell[:, 1], color="black", linewidth=1.1, label="primitive cell")
    ax_real.plot(super_cell[:, 0], super_cell[:, 1], color="tab:gray", linewidth=1.0, linestyle="--", label="supercell")
    _draw_arrow(ax_real, avec[0], "a1", "tab:blue")
    _draw_arrow(ax_real, avec[1], "a2", "tab:orange")
    ax_real.set_title("Direct space")
    ax_real.set_xlabel("x [Angstrom]")
    ax_real.set_ylabel("y [Angstrom]")
    ax_real.set_aspect("equal", adjustable="box")
    ax_real.legend(loc="best", fontsize=8)
    _pad_axes(ax_real, np.vstack([real_cell, super_cell]))

    bz = _wigner_seitz_cell_2d(bvec)
    bz_closed = np.vstack([bz, bz[0]])
    ax_rec.plot(bz_closed[:, 0], bz_closed[:, 1], color="black", linewidth=1.2, label="1st BZ")
    _draw_arrow(ax_rec, bvec[0], "b1", "tab:blue")
    _draw_arrow(ax_rec, bvec[1], "b2", "tab:orange")
    k_cart = _kfrac_to_cart(k_frac, primitive_vectors)[:, :2]
    ax_rec.scatter(k_cart[:, 0], k_cart[:, 1], s=18, alpha=0.35, label="folded k-points")
    if len(path_q):
        path_cart = _kfrac_to_cart(path_q, primitive_vectors)[:, :2]
        ax_rec.scatter(path_cart[:, 0], path_cart[:, 1], s=42, color="tab:red", label="on path")
    nodes = []
    for label in path_labels:
        if label in hs_points:
            point = _kfrac_to_cart(np.asarray([hs_points[label]]), primitive_vectors)[0, :2]
            nodes.append(point)
            ax_rec.text(point[0], point[1], label, ha="center", va="bottom")
    if nodes:
        nodes = np.asarray(nodes)
        ax_rec.plot(nodes[:, 0], nodes[:, 1], color="black", linewidth=1.2, label="path")
    ax_rec.set_title("Reciprocal space")
    ax_rec.set_xlabel("kx [1/Angstrom]")
    ax_rec.set_ylabel("ky [1/Angstrom]")
    ax_rec.set_aspect("equal", adjustable="box")
    ax_rec.legend(loc="best", fontsize=8)
    node_points = np.asarray(nodes) if len(nodes) else bz
    _pad_axes(ax_rec, np.vstack([bz, bvec, k_cart, node_points]))


def _folded_kpoints_from_supercell_matrix(matrix):
    dim = matrix.shape[0]
    det = int(round(abs(np.linalg.det(matrix))))
    inv_t = np.linalg.inv(matrix).T
    points = []
    for indices in np.ndindex(*([det * 4 + 1] * dim)):
        q = inv_t @ np.asarray(indices, dtype=float)
        q = q - np.floor(q)
        if not any(np.allclose(q, old, atol=1e-10) for old in points):
            points.append(q)
        if len(points) == det:
            return np.asarray(points)
    raise RuntimeError("Could not generate folded k-points.")


def _guess_2d_lattice_type(primitive_vectors):
    a = primitive_vectors[0, :2]
    b = primitive_vectors[1, :2]
    la = np.linalg.norm(a)
    lb = np.linalg.norm(b)
    angle = np.degrees(np.arccos(np.clip(np.dot(a, b) / (la * lb), -1.0, 1.0)))
    equal = abs(la - lb) <= 1.0e-3 * max(la, lb)
    if equal and min(abs(angle - 60.0), abs(angle - 120.0)) <= 1.0e-2:
        return "hexagonal"
    if equal and abs(angle - 90.0) <= 1.0e-2:
        return "square"
    if abs(angle - 90.0) <= 1.0e-2:
        return "rectangular"
    return "oblique"


def _standard_kpath(dim, lattice_type, primitive_vectors):
    if dim == 1:
        return {"G": np.array([0.0]), "X": np.array([0.5])}, ["G", "X"]
    lattice_type = _guess_2d_lattice_type(primitive_vectors) if lattice_type == "auto" else lattice_type
    if lattice_type == "hexagonal":
        return {
            "G": np.array([0.0, 0.0]),
            "K": np.array([2.0 / 3.0, 1.0 / 3.0]),
            "M": np.array([0.5, 0.0]),
        }, ["G", "K", "M", "G"]
    return {
        "G": np.array([0.0, 0.0]),
        "X": np.array([0.5, 0.0]),
        "S": np.array([0.5, 0.5]),
        "Y": np.array([0.0, 0.5]),
        "M": np.array([0.5, 0.5]),
    }, ["G", "X", "S", "Y", "G"]


def _parse_path_labels(text, default_path):
    clean = text.strip().replace("Γ", "G")
    if not clean:
        return default_path
    if "-" in clean or "," in clean or " " in clean:
        return [x for x in clean.replace(",", "-").replace(" ", "-").split("-") if x]
    return list(clean)


def _kpath_axis(points, path, primitive_vectors):
    nodes_frac = [points[label] for label in path]
    nodes_cart = _kfrac_to_cart(np.asarray(nodes_frac), primitive_vectors)
    x = [0.0]
    for i in range(1, len(nodes_cart)):
        x.append(x[-1] + float(np.linalg.norm(nodes_cart[i] - nodes_cart[i - 1])))
    return np.asarray(x), nodes_cart


def _project_kpoints_to_path(k_frac_points, points, path, primitive_vectors, tol=1.0e-6):
    dim = k_frac_points.shape[1]
    x_nodes, cart_nodes = _kpath_axis(points, path, primitive_vectors)
    shifts = [np.asarray(s, dtype=float) - 1.0 for s in np.ndindex(*([3] * dim))]
    projected = []
    for ik, q in enumerate(k_frac_points):
        best = None
        for shift in shifts:
            q_equiv = q + shift
            q_cart = _kfrac_to_cart(np.asarray([q_equiv]), primitive_vectors)[0]
            for iseg in range(len(path) - 1):
                a = cart_nodes[iseg]
                b = cart_nodes[iseg + 1]
                v = b - a
                denom = float(np.dot(v, v))
                if denom == 0.0:
                    continue
                t = float(np.dot(q_cart - a, v) / denom)
                if t < -1.0e-10 or t > 1.0 + 1.0e-10:
                    continue
                dist = float(np.linalg.norm(q_cart - (a + t * v)))
                if dist <= tol:
                    x = x_nodes[iseg] + t * float(np.linalg.norm(v))
                    candidate = (dist, ik, x, q_equiv)
                    if best is None or candidate[0] < best[0]:
                        best = candidate
        if best is not None:
            projected.append(best)
    indices = np.asarray([p[1] for p in projected], dtype=int)
    q_equiv = np.asarray([p[3] for p in projected], dtype=float) if projected else np.empty((0, dim))
    return indices, q_equiv


def _update_unfolding_preview(_=None):
    with unfolding_kmap_output:
        unfolding_kmap_output.clear_output(wait=True)
    try:
        if not structure_selector.structure_node:
            unfolding_preview.value = "Select a structure to preview the exact tiling."
            return
        approx = _read_unfolding_vectors()
        dim = approx.shape[0]
        supercell = np.asarray(structure_selector.structure_node.cell, dtype=float)[:dim]
        correction, matrix, snapped, mf = _snap_vectors(approx, supercell)
        points, default_path = _standard_kpath(dim, unfolding_lattice_type.value, snapped)
        path_labels = _parse_path_labels(unfolding_path.value, default_path)
        missing = [label for label in path_labels if label not in points]
        if missing:
            raise ValueError(f"Path labels not available: {missing}")
        k_frac = _folded_kpoints_from_supercell_matrix(matrix)
        path_indices, path_q = _project_kpoints_to_path(k_frac, points, path_labels, snapped)
        labels = ", ".join(points)
        mapping_summary = _mapping_displacement_summary(
            structure_selector.structure_node, snapped, unfolding_primitive_basis_atoms.value
        )
        mapping_text = ""
        if mapping_summary is not None:
            mapping_text = (
                "\n\nPrimitive basis atoms = "
                + " ".join(str(x) for x in mapping_summary["basis_indices"])
                + f"\natom displacement max = {mapping_summary['max']:.6g} Angstrom"
                + f"\natom displacement mean = {mapping_summary['mean']:.6g} Angstrom"
                + f"\nworst atom = {mapping_summary['worst_atom']}"
                + "\nbasis counts = " + str(mapping_summary["basis_counts"].tolist())
            )
        unfolding_preview.value = (
            "Exact primitive vectors used:\n"
            + "\n".join(_vector_to_text(row) for row in snapped)
            + f"\n\nM =\n{matrix}\n"
            + f"det(M) = {int(round(abs(np.linalg.det(matrix))))}\n"
            + f"correction = {correction:.6g} Angstrom\n"
            + f"folded k-points on path = {len(path_indices)} of {len(k_frac)}\n"
            + f"available standard labels: {labels}"
            + mapping_text
        )
        with unfolding_kmap_output:
            if dim == 1:
                fig, ax = plt.subplots(figsize=(5.2, 1.4))
                all_k = _canonical_frac(k_frac)
                ax.scatter(all_k[:, 0], np.zeros(len(all_k)), s=24, alpha=0.5)
                if len(path_q):
                    on_path = _canonical_frac(path_q)
                    ax.scatter(on_path[:, 0], np.zeros(len(on_path)), s=48, color="tab:red")
                ax.set_yticks([])
                ax.set_xlabel("primitive reciprocal fractional k")
                ax.set_xlim(-0.03, 1.03)
                fig.tight_layout()
                plt.show()
            elif dim == 2:
                fig_real, ax_real = plt.subplots(figsize=(7.2, 4.2))
                fig_rec, ax_rec = plt.subplots(figsize=(7.2, 5.2))
                _plot_lattice_diagnostics(ax_real, ax_rec, snapped, supercell, points, path_labels, k_frac, path_q)
                fig_real.tight_layout()
                fig_rec.tight_layout()
                plt.show(fig_real)
                plt.show(fig_rec)
    except Exception as exc:
        unfolding_preview.value = f"Cannot preview unfolding vectors yet: {exc}"


def _update_sparse_retrieve_options(_=None):
    if retrieve_sparse_overlap.value:
        sparse_retrieve_options.children = [overlap_threshold, sparse_overlap_code]
    elif compute_unfolding.value:
        sparse_retrieve_options.children = [overlap_threshold]
    else:
        sparse_retrieve_options.children = []


def _update_unfolding_help(_=None):
    unfolding_help_box.children = [unfolding_help] if unfolding_help_toggle.value else []


def _sync_option_state(_=None):
    if compute_bader_charges.value:
        compute_unfolding.value = False
        write_overlap_matrix.value = False
        retrieve_sparse_overlap.value = False
    elif compute_unfolding.value:
        write_overlap_matrix.value = True

    write_overlap_matrix.disabled = compute_bader_charges.value or compute_unfolding.value
    compute_unfolding.disabled = compute_bader_charges.value

    bader_options.children = [bader_cutoff, bader_code] if compute_bader_charges.value else []
    unfolding_options.children = (
        [
            ipw.HTML("<b>Primitive vectors in Angstrom</b>"),
            unfolding_help_toggle,
            unfolding_help_box,
            unfolding_a1,
            unfolding_a2,
            unfolding_a3,
            ipw.HBox([unfolding_lattice_type, unfolding_path, unfolding_basis_cluster_tol]),
            unfolding_primitive_basis_atoms,
            ipw.HTML("<b>Unfolding code and resources</b>"),
            unfolding_code,
            unfolding_resources,
            unfolding_preview,
            unfolding_kmap_output,
        ]
        if compute_unfolding.value and not compute_bader_charges.value
        else []
    )
    needs_diag = (write_overlap_matrix.value or compute_unfolding.value) and not compute_bader_charges.value
    overlap_options.children = (
        [
            diagonalisation_smearing,
            added_mos,
            overlap_ndigits,
            retrieve_sparse_overlap,
            sparse_retrieve_options,
        ]
        if needs_diag
        else []
    )
    _update_sparse_retrieve_options()
    _update_unfolding_preview()

for widget in (unfolding_a1, unfolding_a2, unfolding_a3, unfolding_lattice_type, unfolding_path, unfolding_primitive_basis_atoms):
    widget.observe(_update_unfolding_preview, "value")
unfolding_help_toggle.observe(_update_unfolding_help, "value")
_update_unfolding_help()
try:
    structure_selector.observe(_update_unfolding_preview, "structure_node")
except Exception:
    pass
write_overlap_matrix.observe(_sync_option_state, "value")
retrieve_sparse_overlap.observe(_sync_option_state, "value")
compute_unfolding.observe(_sync_option_state, "value")
compute_bader_charges.observe(_sync_option_state, "value")
_sync_option_state()

parent_calc_folder = ipw.Text(
    value="",
    description="WFN restart folder:",
    placeholder="Optional remote path on the selected computer",
    style={"description_width": "initial"},
    layout={"width": "70%"},
)

advanced_settings = ipw.Accordion(
    children=[ipw.VBox([parent_calc_folder])],
    selected_index=None,
)
advanced_settings.set_title(0, "Advanced settings")







In [ ]:
workflow_description = ipw.Text(
    description="Workflow description:",
    placeholder="Provide the description here.",
    style={"description_width": "initial"},
    layout={"width": "70%"},
)

In [ ]:
ipw.dlink((code_input_widget, "value"), (input_details, "selected_code"))


def prepare_scf():
    with output:
        clear_output()
    if not structure_selector.structure_node:
        can_submit, msg = False, "Select a structure first."
    elif not code_input_widget.value:
        can_submit, msg = False, "Select CP2K code."
    elif compute_bader_charges.value and not bader_code.value:
        can_submit, msg = False, "Select Bader code."
    elif compute_unfolding.value and not unfolding_code.value:
        can_submit, msg = False, "Select unfolding code."
    elif compute_unfolding.value and not unfolding_a1.value.strip():
        can_submit, msg = False, "Enter at least primitive vector a1 for unfolding."
    elif compute_unfolding.value and not unfolding_primitive_basis_atoms.value.strip():
        can_submit, msg = False, "Enter primitive basis atom indices for unfolding."
    elif (
        not compute_bader_charges.value
        and retrieve_sparse_overlap.value
        and not sparse_overlap_code.value
    ):
        can_submit, msg = False, "Select overlap converter code."
    else:
        can_submit, msg, parameters = input_details.return_final_dictionary()
        parent_calc_folder_path = parent_calc_folder.value.strip()

    if not can_submit:
        with output:
            print(msg)
            return

    code = orm.load_code(code_input_widget.value)
    dft_params_dict = parameters["dft_params"]
    dft_params_dict.setdefault("periodic", "XYZ")
    dft_params_dict["elpa_switch"] = True
    dft_params_dict["sc_diag"] = (
        False
        if compute_bader_charges.value
        else diagonalisation_smearing.enable_diagonalisation.value
    )

    if (
        not compute_bader_charges.value
        and (write_overlap_matrix.value or compute_unfolding.value)
        and added_mos.value > 0
    ):
        dft_params_dict["added_mos"] = added_mos.value

    if (
        not compute_bader_charges.value
        and (write_overlap_matrix.value or compute_unfolding.value)
        and diagonalisation_smearing.smearing_enabled
    ):
        dft_params_dict["smear_t"] = diagonalisation_smearing.smearing_temperature.value
        dft_params_dict["force_multiplicity"] = diagonalisation_smearing.force_multiplicity.value

    builder = Cp2kScfWorkChain.get_builder()
    builder.protocol = orm.Str(protocol.value)
    builder.metadata.label = (
        "CP2K_SCF_bader"
        if compute_bader_charges.value
        else "CP2K_SCF_unfolding"
        if compute_unfolding.value
        else "CP2K_SCF_overlap"
        if write_overlap_matrix.value
        else "CP2K_SCF"
    )
    builder.metadata.description = workflow_description.value
    builder.cp2k_code = code
    builder.options = {
        "max_wallclock_seconds": resources.walltime_seconds,
        "resources": {
            "num_machines": resources.nodes,
            "num_mpiprocs_per_machine": resources.tasks_per_node,
            "num_cores_per_mpiproc": resources.threads_per_task,
        },
    }
    builder.structure = structure_selector.structure_node
    builder.dft_params = orm.Dict(dft_params_dict)
    builder.compute_bader_charges = orm.Bool(compute_bader_charges.value)
    builder.write_overlap_matrix = orm.Bool(
        (write_overlap_matrix.value or compute_unfolding.value) and not compute_bader_charges.value
    )
    builder.retrieve_sparse_overlap = orm.Bool(
        retrieve_sparse_overlap.value and not compute_bader_charges.value
    )
    builder.compute_unfolding = orm.Bool(
        compute_unfolding.value and not compute_bader_charges.value
    )
    builder.overlap_ndigits = orm.Int(overlap_ndigits.value)
    builder.overlap_threshold = orm.Float(overlap_threshold.value)
    builder.bader_cutoff = orm.Float(bader_cutoff.value)
    if retrieve_sparse_overlap.value and not compute_bader_charges.value:
        builder.sparse_overlap_code = orm.load_code(sparse_overlap_code.value)
    if compute_unfolding.value and not compute_bader_charges.value:
        builder.unfolding_code = orm.load_code(unfolding_code.value)
        builder.unfolding_primitive_vectors = orm.Str(
            "; ".join(w.value.strip() for w in (unfolding_a1, unfolding_a2, unfolding_a3) if w.value.strip())
        )
        builder.unfolding_primitive_basis_atoms = orm.Str(unfolding_primitive_basis_atoms.value.strip())
        builder.unfolding_path = orm.Str(unfolding_path.value.strip() or "G-K-M-G")
        builder.unfolding_lattice_type = orm.Str(unfolding_lattice_type.value)
        builder.unfolding_basis_cluster_tol = orm.Float(unfolding_basis_cluster_tol.value)
        builder.unfolding_options = orm.Dict({
            "max_wallclock_seconds": unfolding_resources.walltime_seconds,
            "resources": {
                "num_machines": unfolding_resources.nodes,
                "num_mpiprocs_per_machine": unfolding_resources.tasks_per_node,
                "num_cores_per_mpiproc": unfolding_resources.threads_per_task,
            },
        })
    if compute_bader_charges.value:
        builder.bader_code = orm.load_code(bader_code.value)

    if parent_calc_folder_path:
        selected_parent_calc_folder = orm.RemoteData(
            computer=code.computer,
            remote_path=parent_calc_folder_path,
        )
        print(
            "Restarting from selected parent folder: "
            f"{parent_calc_folder_path}"
        )
        builder.parent_calc_folder = selected_parent_calc_folder
    else:
        # Check if a restart wfn is available.
        wave_function = None
        if structure_selector.structure_node.is_stored:
            wave_function = wfn.structure_available_wfn(
                node=structure_selector.structure_node,
                relative_replica_id=None,
                current_hostname=code.computer.hostname,
                return_path=False,
                dft_params=dft_params_dict,
            )
        if wave_function is not None:
            print(f"Restarting from wfn in folder: {wave_function.pk}")
            builder.parent_calc_folder = wave_function

    return builder



In [ ]:
btn_submit = awb.SubmitButtonWidget(
    Cp2kScfWorkChain,
    inputs_generator=prepare_scf,
    disable_after_submit=False,
    append_output=True,
)

In [ ]:
# Resources estimation.
resources_estimation = awe.ResourcesEstimatorWidget(
    price_link="https://2go.cscs.ch/offering/swiss_academia/institutional_customers/",
    price_per_hour=2.85,
)
resources_estimation.link_to_resources_widget(resources)
ipw.dlink((input_details, "details"), (resources_estimation, "details"))
ipw.dlink((input_details, "uks"), (resources_estimation, "uks"))
_ = ipw.dlink((code_input_widget, "value"), (resources_estimation, "selected_code"))

# Inputs

In [ ]:
display(
    input_details,
    protocol,
    compute_bader_charges,
    bader_options,
    compute_unfolding,
    unfolding_options,
    write_overlap_matrix,
    overlap_options,
    advanced_settings,
)


# Code and resources

In [ ]:
display(code_input_widget, ipw.HBox([resources, resources_estimation]))

# Submit

In [ ]:
display(workflow_description, btn_submit, output)